B"H

# Milestone 3: Cleaning and Formatting the Website Data

- DSC-540
- David Koyrakh
- Professor Catie Williams

## Load and examine the data

For the website data source, I plan to utilize a specific table in the Wikipedia article on religiosity across the United States. The table displays data from a 2014 Pew Research study and includes state-by-state metrics for belief in G-d, importance of religion, and frequency of prayer.

Wikipedia renders tables using `<table>` HTML tags, so Pandas' own `read_html()` method is able to directly extract the tables from the page:

### Important necessary libraries

In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np

### Perform the extraction

In [3]:
# Store the URL
url = 'https://en.wikipedia.org/wiki/List_of_U.S._states_and_territories_by_religiosity#cite_ref-61'

# Extract tables
tables = pd.read_html(url)

# Store the religosity-by-state table
religiosity_table = tables[5] # 6th table on the page

print('Dataset shape:', religiosity_table.shape)
religiosity_table.head()

Dataset shape: (51, 6)


,State or District,Overall Religiosity Rank,Believe in God with Certainty,Consider Religion Important,Pray Daily,Attend Weekly Worship Services
0,California,35,54%,47%,51%,31%
1,Texas,11,69%,63%,63%,42%
2,Florida,22,64%,53%,56%,35%
3,New York,43,56%,45%,48%,29%
4,Illinois,33,61%,50%,51%,34%


We can also take a look at the _end_ of the table:

In [4]:
religiosity_table.tail()

,State or District,Overall Religiosity Rank,Believe in God with Certainty,Consider Religion Important,Pray Daily,Attend Weekly Worship Services
46,Alaska,44,55%,41%,49%,30%
47,North Dakota,27,64%,53%,51%,33%
48,District of Columbia,27,55%,50%,51%,28%
49,Vermont,48,41%,32%,33%,21%
50,Wyoming,22,66%,49%,53%,38%


A manual check of these values confirms that the extracted tabular data is exactly the same as what is visually rendered by the browser. The scraping and extraction was succesful.

### Ensure that all states are represented equally

It's important to understand the dataset well. In this case, I want to ensure that all states are present one time:

In [5]:
# Count number of rows
print("Table rows:", len(religiosity_table))

# Count unique values for the columns: State, Area_Name, and Attribute
print("Unique 'State' values:", religiosity_table['State or District'].nunique())

Table rows: 51
Unique 'State' values: 51


The above output shows that the length of this dataset is 51 (including all US states and Washington, DC) and that each row is unique. This confirms that the dataset contains no duplicates and is one state per row. This indeed matches what we see visually rendered on the Wikipedia page.

## Step 1: Rename column names to be more respectful of religious sensitivities

Currently, the column headers which refer to G-d spell out His Name without a hyphen, which is not considered respectful by certain religious groups. In order to make this dataset as useful and ethical as possible, I will rename any columns containing G-d's name:

In [6]:
# Replace any occurrence of 'G-d' in column names dynamically, using regex
religiosity_table.rename(columns=lambda x: x.replace(x[x.find("G"):x.find("d")+1], "G-d") if "G" in x and "d" in x else x, inplace=True)

In [7]:
religiosity_table.columns.values

array(['State or District', 'Overall Religiosity Rank',
       'Believe in G-d with Certainty', 'Consider Religion Important',
       'Pray Daily', 'Attend Weekly Worship Services'], dtype=object)

The column which previously contained the word 'G-d' without a hyphen has been corrected.

## Step 2: Create an abbreviated State column

In this step, I will create a normalized `State` column. Since, ultimately, this data will be merged and visualized together with national data from other sources, it is important to have a set of state keys which is shared by all of the datasets. I will continue to use the abbreviated state/region format that was used in Milestone 1. I will name this new column, "State", rather than "State (abbrev)" or the like, in order to make it most human-readable and also match the name with the State ID column of the dataset from Milestone 1. To this end, I will now create a new column in this dataset called 'State' with the abbreviated state/region for each row.

### First, define a dictionary mapping each state/region (as it features in raw table data) to the state ID.

In [8]:
state_abbr = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA',
    'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE', 'District of Columbia': 'DC', 'Florida': 'FL',
    'Georgia': 'GA', 'Hawaii': 'HI', 'Idaho': 'ID', 'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA',
    'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD', 'Massachusetts': 'MA',
    'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS', 'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE',
    'Nevada': 'NV', 'New Hampshire': 'NH', 'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY',
    'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK', 'Oregon': 'OR',
    'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC', 'South Dakota': 'SD', 'Tennessee': 'TN',
    'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV',
    'Wisconsin': 'WI', 'Wyoming': 'WY'
}

### Next, create the new, normalized `State` column:

In [9]:
religiosity_table["State"] = religiosity_table["State or District"].map(state_abbr)

# Display the new column alongside 'State or District' to verify results
religiosity_table[["State or District", "State"]]

,State or District,State
0,California,CA
1,Texas,TX
2,Florida,FL
3,New York,NY
4,Illinois,IL
5,Pennsylvania,PA
6,Ohio,OH
7,Georgia,GA
8,Michigan,MI
9,North Carolina,NC


Success! A manual inspection of the previous cell's output shows that the normalized `State` column was created successfully. Even 'District of Columbia' has been mapped correctly to its abbreviated State ID ('DC').

## Step 3: Ensure proper types

Since the data was extracted from a Wikipedia page and contains characters such as %, we must validate the data types to ensure that they are numeric.

First, we must check the data types of the stored table:

In [10]:
print('Dataset types:')
print(religiosity_table.info())


Dataset types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 7 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   State or District               51 non-null     object
 1   Overall Religiosity Rank        51 non-null     int64 
 2   Believe in G-d with Certainty   51 non-null     object
 3   Consider Religion Important     51 non-null     object
 4   Pray Daily                      51 non-null     object
 5   Attend Weekly Worship Services  51 non-null     object
 6   State                           51 non-null     object
dtypes: int64(1), object(6)
memory usage: 2.9+ KB
None


Indeed, it appears that only `Overall Religiosity Rank` was correctly parsed as an integer. The remaining columns, however, show `object`, meaning they are not strictly numeric. The most likely culprit is the presence of a special character (the `%` sign) in these columns. The percentage data must be parsed into floats representing percentage values as decimals:

In [11]:
# First, list all of the columns
religiosity_table.columns.values

array(['State or District', 'Overall Religiosity Rank',
       'Believe in G-d with Certainty', 'Consider Religion Important',
       'Pray Daily', 'Attend Weekly Worship Services', 'State'],
      dtype=object)

In [12]:
# Define array of columns containing percentages
percentage_columns = ["Believe in G-d with Certainty", "Consider Religion Important", "Pray Daily", "Attend Weekly Worship Services"]

# Apply the transformation to each column
for col in percentage_columns:
    religiosity_table[col] = religiosity_table[col].astype(str).str.rstrip('%').astype(float) / 100

religiosity_table.head()

,State or District,Overall Religiosity Rank,Believe in G-d with Certainty,Consider Religion Important,Pray Daily,Attend Weekly Worship Services,State
0,California,35,0.54,0.47,0.51,0.31,CA
1,Texas,11,0.69,0.63,0.63,0.42,TX
2,Florida,22,0.64,0.53,0.56,0.35,FL
3,New York,43,0.56,0.45,0.48,0.29,NY
4,Illinois,33,0.61,0.50,0.51,0.34,IL


These converted float values correctly match the percentage values that we saw earlier for each column. Now, let's double-check to ensure that the data types of all columns are now numeric (besides for State and State ID).

In [13]:
religiosity_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 7 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   State or District               51 non-null     object 
 1   Overall Religiosity Rank        51 non-null     int64  
 2   Believe in G-d with Certainty   51 non-null     float64
 3   Consider Religion Important     51 non-null     float64
 4   Pray Daily                      51 non-null     float64
 5   Attend Weekly Worship Services  51 non-null     float64
 6   State                           51 non-null     object 
dtypes: float64(4), int64(1), object(2)
memory usage: 2.9+ KB


All numeric columns are now of the correct type.

## Step 4: Create a row for US-aggregate data

As with our flat file data from Milestone 1, it would be helpful for this dataset to feature a row for US-aggregate data. However, that is not provided by the raw data from this table; we will need to calculate it. Since this is aggregate data from all the states, the `Overall Religiosity Rank` column isn't meaningful for this new row; we will leave it blank (NaN).

This new row will contain the US-aggregate data for all remaining rows, which contain percentage values. Therefore, the calculation for each column is simply taking the mean of all existing rows per column:

In [14]:
# Generate a row from the mean of each column, besides the State and Rank columns.
# We can use our 'percentage_columns' array to target the correct columns
us_aggregate = religiosity_table[percentage_columns].mean().round(2) # round to the nearest hundredth
us_aggregate

Believe in G-d with Certainty     0.63
Consider Religion Important       0.53
Pray Daily                        0.54
Attend Weekly Worship Services    0.36
dtype: float64

Now the `us_aggregate` row can be finalized and appended to our dataset:

In [15]:
# Prepare the `us_aggregate` row for insertion
us_aggregate_row = pd.DataFrame([{
    "State or District": "United States",
    "State": "US",
    "Overall Religiosity Rank": np.nan,  # Leave blank as it's not meaningful
    "Believe in G-d with Certainty": us_aggregate["Believe in G-d with Certainty"],
    "Consider Religion Important": us_aggregate["Consider Religion Important"],
    "Pray Daily": us_aggregate["Pray Daily"],
    "Attend Weekly Worship Services": us_aggregate["Attend Weekly Worship Services"]
}])
us_aggregate_row

,State or District,State,Overall Religiosity Rank,Believe in G-d with Certainty,Consider Religion Important,Pray Daily,Attend Weekly Worship Services
0,United States,US,NaN,0.63,0.53,0.54,0.36


In [16]:
# Add the new row to our dataset:
religiosity_table = pd.concat([religiosity_table, us_aggregate_row], ignore_index=True)
religiosity_table.tail()

,State or District,Overall Religiosity Rank,Believe in G-d with Certainty,Consider Religion Important,Pray Daily,Attend Weekly Worship Services,State
47,North Dakota,27.0,0.64,0.53,0.51,0.33,ND
48,District of Columbia,27.0,0.55,0.50,0.51,0.28,DC
49,Vermont,48.0,0.41,0.32,0.33,0.21,VT
50,Wyoming,22.0,0.66,0.49,0.53,0.38,WY
51,United States,NaN,0.63,0.53,0.54,0.36,US


The US aggregate row has been successfully added to our data.

## Step 5: Create a new column for states' overall `Religiosity Score`

The Pew Research/Wikipedia data is provided with an `Overall Religiosity Rank` column, but it isn't immediately clear how it was computed and exactly what it measures. In fact, it does not seem to be a typical national/state-by-state rating, since there are many duplicate values:

In [17]:
# Count duplicates in the "Overall Religiosity Rank" column
duplicate_ranks_count = religiosity_table["Overall Religiosity Rank"].duplicated().sum()
print("Duplicates in `Overall Religiosity Rank`:", duplicate_ranks_count)

Duplicates in `Overall Religiosity Rank`: 24


In order to improve this dataset's readability, I propose replacing the provided `Overall Religiosity Rank` column with a newly-computed column called `Religiosity Score`. It can be conveniently computed by taking the mean of the following columns:

- Believe in G-d with Certainty
- Consider Religion Important
- Pray Daily
- Attend Weekly Worship Services

... Since each of those columns are on the same scale (percentage of population, in decimal-form), direction (higher = more religious) and share the same data type (`float`).

In [18]:
# Compute a new "Religiosity Score" based on the mean of religiosity metrics
religiosity_table["Religiosity Score"] = religiosity_table[[
       'Believe in G-d with Certainty', 'Consider Religion Important',
       'Pray Daily', 'Attend Weekly Worship Services']].mean(axis=1)

# Preview the updated Religosity Table, sorted by the new `Religiosity Score` column
religiosity_table_sorted = religiosity_table.sort_values(by="Religiosity Score", ascending=False)
religiosity_table_sorted.head(10)

,State or District,Overall Religiosity Rank,Believe in G-d with Certainty,Consider Religion Important,Pray Daily,Attend Weekly Worship Services,State,Religiosity Score
23,Alabama,1.0,0.82,0.77,0.73,0.51,AL,0.7075
30,Mississippi,1.0,0.82,0.74,0.75,0.49,MS,0.7000
16,Tennessee,3.0,0.78,0.71,0.70,0.51,TN,0.6750
24,Louisiana,4.0,0.75,0.71,0.68,0.46,LA,0.6500
22,South Carolina,5.0,0.74,0.69,0.66,0.47,SC,0.6400
37,West Virginia,7.0,0.77,0.64,0.68,0.46,WV,0.6375
31,Arkansas,5.0,0.77,0.70,0.65,0.41,AR,0.6325
7,Georgia,8.0,0.74,0.64,0.64,0.42,GA,0.6100
27,Oklahoma,8.0,0.71,0.64,0.65,0.43,OK,0.6075
9,North Carolina,10.0,0.73,0.62,0.66,0.39,NC,0.6000


Coincidently, the new `Religosity Score` column mostly (but not exactly) coincides with the provided `Overall Religiosity Rank`. An added bonus of the new column is that the way it is calculated is entirely documented and transparent, right here in this notebook. Therefore, it can be more readily used, understood, and applied within reasonable limits.

Lastly, let's remove the old `Religosity Score` column:

In [19]:
# Drop `Overall Religiosity Rank` column
religiosity_table_sorted.drop(columns=["Overall Religiosity Rank"], inplace=True)

Here is the full, final dataset:

In [20]:
religiosity_table_sorted

,State or District,Believe in G-d with Certainty,Consider Religion Important,Pray Daily,Attend Weekly Worship Services,State,Religiosity Score
23,Alabama,0.82,0.77,0.73,0.51,AL,0.7075
30,Mississippi,0.82,0.74,0.75,0.49,MS,0.7000
16,Tennessee,0.78,0.71,0.70,0.51,TN,0.6750
24,Louisiana,0.75,0.71,0.68,0.46,LA,0.6500
22,South Carolina,0.74,0.69,0.66,0.47,SC,0.6400
37,West Virginia,0.77,0.64,0.68,0.46,WV,0.6375
31,Arkansas,0.77,0.70,0.65,0.41,AR,0.6325
7,Georgia,0.74,0.64,0.64,0.42,GA,0.6100
27,Oklahoma,0.71,0.64,0.65,0.43,OK,0.6075
9,North Carolina,0.73,0.62,0.66,0.39,NC,0.6000


## Conclusion

The transformations in this notebook included:

1. Renaming column names to be more respectful to accommodate various religious sensitivities.
2. Creating an abbreviated State column – Added a standardized two-letter state abbreviation for consistency and easier filtering.
3. Ensuring proper data types – Converted percentage strings (e.g., "54%") into decimal values (e.g., 0.54) to allow numerical analysis.
4. Creating a row for US-aggregate data – Calculated a national summary row by averaging percentage values across all states.
5. Creating a new `Religosity Score` column – a new ranking system was created to clearly represent and compare state-level religiosity.

This dataset was sourced from Wikipedia, which cites the Pew Research Center as its original source. Since it is publicly available, there are no direct legal restrictions on its use, but ethical considerations must be taken into account. One risk of transforming this dataset is that aggregating data may misrepresent regional variations, especially when computing the national average. Additionally, renaming columns and replacing the original ranking column affects the data's narrative, which can potenitally lead to unintended bias.

I made several assumptions while cleaning this data:

- The dataset was correctly sourced from Pew Research by Wikipedia.
- The Pew Research study is a current and reliable source of information.
- 'G-d' is the most respectful way of writing G-d's Name (Step 1).

The dataset was obtained ethically from a publicly available source and was not manipulated in a way that misrepresents the original data. However, to mitigate risks, transparency is key. Any future use of this dataset should include clear documentation of modifications, especially regarding ranking changes and aggregation methods, to avoid misleading conclusions.